# SwinUNETR Training — RGB PlanetScope Tiles

This notebook trains **SwinUNETR from MONAI** for binary glacial lake segmentation using the same general flow as your UNet++ notebook:

- paired image/mask GeoTIFF discovery
- RGB band selection
- per-band min-max normalization
- BCE + Dice loss
- IoU, precision, recall, F1, accuracy logging
- best model checkpoint by validation IoU

> Note: Unlike `segmentation_models_pytorch` encoders, MONAI `SwinUNETR` does **not** provide a simple `encoder_weights="imagenet"` argument. This notebook trains SwinUNETR from scratch.

## Dependencies

In [ ]:
# Run this once if needed.
# %pip install -q torch torchvision monai rasterio albumentations pandas numpy

## Configuration

In [1]:
from pathlib import Path
import random
import torch

IMAGES_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\subset_750\images")
LABELS_DIR = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\subset_750\labels")

# Your RGB outputs appear to be 3-band GeoTIFFs, so use bands 1,2,3.
# If you are reading directly from 8-band Planet SuperDove SR, use RGB_BANDS = (6, 4, 2).
RGB_BANDS = (3, 2, 1)

RUN_NAME = "rgb_dataset_10_epochs"
OUTPUT_DIR_LOGS = Path(r"D:\Thesis\glacial_lake_detection_thesis\experiments\logs\SwinUNETR")
OUTPUT_DIR_MODELS = Path(r"D:\Thesis\glacial_lake_detection_thesis\models\saved_models\SwinUNETR")
(OUTPUT_DIR_LOGS / RUN_NAME).mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR_MODELS / RUN_NAME).mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_LOGS = OUTPUT_DIR_LOGS / RUN_NAME
OUTPUT_DIR_MODELS = OUTPUT_DIR_MODELS / RUN_NAME

NUM_SAMPLES = 750
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-4

# SwinUNETR settings.
# TILE_SIZE should match your image tile size. Your previous note mentioned 512x512 tiles.
TILE_SIZE = 512
IN_CHANNELS = 3
NUM_CLASSES = 1
FEATURE_SIZE = 24       # use 24 for lower VRAM; 48 is heavier
USE_CHECKPOINT = True   # gradient checkpointing saves VRAM

# Optional: use smaller tiles if you get CUDA out-of-memory, e.g. TILE_SIZE = 256
# but then your dataset must resize/crop to that size.

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

## Discover Pairs & Sample

In [2]:
def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} pairs.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

Found 750 pairs.
Train: 600, Val: 150


## Dataset & Dataloaders

In [3]:
import numpy as np
import rasterio
import albumentations as A
from torch.utils.data import Dataset, DataLoader

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C,H,W)
    return arr

def select_bands(x: np.ndarray, bands=(3,2,1)):
    idx = [b - 1 for b in bands]
    if max(idx) >= x.shape[0]:
        raise ValueError(
            f"Requested bands {bands}, but image only has {x.shape[0]} band(s). "
            "For 3-band RGB GeoTIFFs, use RGB_BANDS=(1,2,3) or adjust based on your band order."
        )
    return x[idx, ...].astype(np.float32)

def minmax_per_band(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    for i in range(x.shape[0]):
        vmin = np.min(x[i])
        vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTilesRGB(Dataset):
    def __init__(self, pairs, bands=(3,2,1), aug=None, normalize=True):
        self.pairs = pairs
        self.bands = bands
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img_all = read_geotiff(img_path)
        img = select_bands(img_all, self.bands)  # (3,H,W)

        with rasterio.open(lbl_path) as src:
            lbl = src.read(1)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        img_hwc = np.transpose(img, (1, 2, 0))  # (H,W,C)
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        img = torch.from_numpy(np.transpose(img_hwc, (2, 0, 1))).float()
        lbl = torch.from_numpy(lbl_hwc[..., 0]).float()
        return img, lbl

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
])

val_aug = A.Compose([])

train_ds = LakeTilesRGB(train_pairs, bands=RGB_BANDS, aug=None, normalize=True)
val_ds = LakeTilesRGB(val_pairs, bands=RGB_BANDS, aug=None, normalize=True)

PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

len(train_ds), len(val_ds)

c:\Users\gg\anaconda3\envs\deeplearning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(600, 150)

## Loss & Metrics

In [4]:
import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    if probs.dim() == 4:
        probs = probs[:, 0]
    intersection = (probs * targets).sum(dim=(1, 2))
    union = probs.sum(dim=(1, 2)) + targets.sum(dim=(1, 2)) + eps
    dice = (2 * intersection + eps) / union
    return 1 - dice.mean()

def bce_dice_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_with_logits(logits, targets)

def compute_metrics_from_logits(logits, targets, threshold=0.5):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        if probs.dim() == 4:
            probs = probs[:, 0]
        preds = (probs > threshold).float()
        t = targets.float()

        tp = (preds * t).sum(dim=(1, 2))
        tn = ((1 - preds) * (1 - t)).sum(dim=(1, 2))
        fp = (preds * (1 - t)).sum(dim=(1, 2))
        fn = ((1 - preds) * t).sum(dim=(1, 2))

        eps = 1e-7
        precision = (tp / (tp + fp + eps)).mean().item()
        recall = (tp / (tp + fn + eps)).mean().item()
        f1 = (2 * precision * recall) / (precision + recall + eps)
        iou = (tp / (tp + fp + fn + eps)).mean().item()
        accuracy = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()

        return {
            "iou": iou,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": accuracy,
        }

## Model — MONAI SwinUNETR

In [5]:
import inspect
import pandas as pd
from datetime import datetime
from torch import optim
from monai.networks.nets import SwinUNETR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def make_swinunetr_2d(
    img_size=512,
    in_channels=3,
    out_channels=1,
    feature_size=24,
    use_checkpoint=True,
):
    """
    Build MONAI SwinUNETR for 2D binary segmentation.

    MONAI versions differ slightly:
    - some versions require img_size
    - newer versions may deprecate img_size but still accept it
    This function checks the installed signature and passes compatible arguments.
    """
    kwargs = dict(
        in_channels=in_channels,
        out_channels=out_channels,
        feature_size=feature_size,
        use_checkpoint=use_checkpoint,
        spatial_dims=2,
    )

    sig = inspect.signature(SwinUNETR)
    if "img_size" in sig.parameters:
        kwargs["img_size"] = (img_size, img_size)

    model = SwinUNETR(**kwargs)
    return model

def run_one_epoch(model, loader, optimizer=None, use_amp=True):
    use_amp = bool(use_amp and torch.cuda.is_available())
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    n_batches = 0
    agg = dict(iou=0.0, precision=0.0, recall=0.0, f1=0.0, accuracy=0.0)

    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            if logits.dim() == 4:
                logits = logits[:, 0]  # (B,H,W)
            loss = bce_dice_loss(logits, lbls)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item()
        n_batches += 1

        m = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg:
            agg[k] += m[k]

    avg_loss = total_loss / max(1, n_batches)
    for k in agg:
        agg[k] /= max(1, n_batches)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return avg_loss, agg

# Quick shape test before full training
model_test = make_swinunetr_2d(
    img_size=TILE_SIZE,
    in_channels=IN_CHANNELS,
    out_channels=NUM_CLASSES,
    feature_size=FEATURE_SIZE,
    use_checkpoint=USE_CHECKPOINT,
).to(device)

x = torch.randn(1, IN_CHANNELS, TILE_SIZE, TILE_SIZE).to(device)
with torch.no_grad():
    y = model_test(x)
print("Input shape:", tuple(x.shape), "Output shape:", tuple(y.shape))
del model_test, x, y
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cuda
Input shape: (1, 3, 512, 512) Output shape: (1, 1, 512, 512)


## Training

In [6]:
results_csv = OUTPUT_DIR_LOGS / "results.csv"

if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp",
        "model",
        "epoch",
        "split",
        "loss",
        "iou",
        "precision",
        "recall",
        "f1",
        "accuracy",
    ]).to_csv(results_csv, index=False)

model_name = f"SwinUNETR_feature{FEATURE_SIZE}_tile{TILE_SIZE}"
print(f"\n=== Training model: {model_name} ===")

model = make_swinunetr_2d(
    img_size=TILE_SIZE,
    in_channels=IN_CHANNELS,
    out_channels=NUM_CLASSES,
    feature_size=FEATURE_SIZE,
    use_checkpoint=USE_CHECKPOINT,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val_iou = -1.0
best_path = OUTPUT_DIR_MODELS / f"{model_name}_best_val_iou.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer, use_amp=True)
    val_loss, val_metrics = run_one_epoch(model, val_loader, optimizer=None, use_amp=True)

    print(
        f"[{model_name}] Epoch {epoch}/{EPOCHS} — "
        f"train IoU={train_metrics['iou']:.4f}  val IoU={val_metrics['iou']:.4f}  "
        f"train Loss={train_loss:.4f}  val Loss={val_loss:.4f}"
    )

    ts = datetime.utcnow().isoformat()

    pd.DataFrame([
        dict(timestamp=ts, model=model_name, epoch=epoch, split="train", loss=train_loss, **train_metrics),
        dict(timestamp=ts, model=model_name, epoch=epoch, split="val", loss=val_loss, **val_metrics),
    ]).to_csv(results_csv, mode="a", index=False, header=False)

    if val_metrics["iou"] > best_val_iou:
        best_val_iou = val_metrics["iou"]
        torch.save({
            "model": model_name,
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_iou": best_val_iou,
            "config": {
                "tile_size": TILE_SIZE,
                "in_channels": IN_CHANNELS,
                "num_classes": NUM_CLASSES,
                "feature_size": FEATURE_SIZE,
                "use_checkpoint": USE_CHECKPOINT,
                "rgb_bands": RGB_BANDS,
            },
        }, best_path)
        print(f"Saved best model: {best_path} | best_val_iou={best_val_iou:.4f}")

print(f"Training complete. Logs: {results_csv}")
print(f"Best model saved at: {best_path}")


=== Training model: SwinUNETR_feature24_tile512 ===


C:\Users\gg\AppData\Local\Temp\ipykernel_22092\1633076896.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\gg\AppData\Local\Temp\ipykernel_22092\1633076896.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[SwinUNETR_feature24_tile512] Epoch 1/10 — train IoU=0.3069  val IoU=0.3913  train Loss=1.2397  val Loss=1.0872
Saved best model: D:\Thesis\glacial_lake_detection_thesis\models\saved_models\SwinUNETR\rgb_dataset_10_epochs\SwinUNETR_feature24_tile512_best_val_iou.pt | best_val_iou=0.3913


C:\Users\gg\AppData\Local\Temp\ipykernel_22092\3045761346.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat()


[SwinUNETR_feature24_tile512] Epoch 2/10 — train IoU=0.4261  val IoU=0.5321  train Loss=1.0370  val Loss=0.9541
Saved best model: D:\Thesis\glacial_lake_detection_thesis\models\saved_models\SwinUNETR\rgb_dataset_10_epochs\SwinUNETR_feature24_tile512_best_val_iou.pt | best_val_iou=0.5321
[SwinUNETR_feature24_tile512] Epoch 3/10 — train IoU=0.4499  val IoU=0.5224  train Loss=0.9407  val Loss=0.8687
[SwinUNETR_feature24_tile512] Epoch 4/10 — train IoU=0.4819  val IoU=0.5613  train Loss=0.8678  val Loss=0.7987
Saved best model: D:\Thesis\glacial_lake_detection_thesis\models\saved_models\SwinUNETR\rgb_dataset_10_epochs\SwinUNETR_feature24_tile512_best_val_iou.pt | best_val_iou=0.5613
[SwinUNETR_feature24_tile512] Epoch 5/10 — train IoU=0.4989  val IoU=0.4992  train Loss=0.8001  val Loss=0.7428
[SwinUNETR_feature24_tile512] Epoch 6/10 — train IoU=0.5222  val IoU=0.5872  train Loss=0.7366  val Loss=0.6684
Saved best model: D:\Thesis\glacial_lake_detection_thesis\models\saved_models\SwinUNETR\

## Notes for CUDA Out-of-Memory

In [ ]:
# If you get CUDA OOM:
# 1. Keep BATCH_SIZE = 1 or 2
# 2. Keep FEATURE_SIZE = 24 instead of 48
# 3. Keep USE_CHECKPOINT = True
# 4. Use TILE_SIZE = 256 only if your images/masks are resized or cropped to 256x256
# 5. Close other GPU processes before training